# 33 · 检索评估：Recall / MRR / NDCG

> 调 chunk、top-k、索引之前，先学会**量化检索好不好**。否则一切“感觉好”都是幻觉。

**本文件覆盖知识点**：Recall / Precision / Hit Rate / Recall@K / Precision@K / MRR / NDCG / MAP

先准备一份“测试集 + 人工标注的相关文档”，这是评估的前提。

In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集.md 是「人工标注的答案」，不能进索引 —— 否则第 33 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里显式排除。
_EXCLUDE = {'评测集.md'}

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name in _EXCLUDE:
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


In [ ]:
# 评测集：data/评测集.md 是人工标注的 ground truth（标注文件本身不进索引，否则就是把答案当资料，指标虚高）
def load_eval_set(path='data/评测集.md'):
    """解析标注表：一行 = 一个问题 + 它「应当被检索出来」的(文档, 小节)"""
    items = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        m = re.match(r'^\|\s*(\d+)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|$', line)
        if m:
            items.append({'q': m.group(2), 'doc': m.group(3), 'section': m.group(4)})
    return items

test_set = load_eval_set()
for t in test_set:      # 标注粒度是「小节」，展开成 chunk id 集合：同一小节切出的片段都算相关
    t['rel'] = sorted(c['i'] for c in CHUNKS if c['source'] == t['doc'] and c['section'] == t['section'])

missing = [t['q'] for t in test_set if not t['rel']]
assert not missing, '标注指向了语料里不存在的小节，需修标注或补语料：%s' % missing
print('评测集：%d 个问题，标注相关片段 %d 个（语料共 %d 个片段）'
      % (len(test_set), sum(len(t['rel']) for t in test_set), len(CHUNKS)))
print('例：', test_set[0]['q'], '→', test_set[0]['doc'], test_set[0]['section'], '→ chunk', test_set[0]['rel'])

## 先用大白话讲一遍（公式看不懂就先看这里）

一次检索只要数出三个数，指标基本就是小学算术：

- **该找的** = 人工标注为相关的文档（记作 G），比如 1 条；
- **找到的** = 系统返回的前 K 条（记作 R_K），比如 3 条；
- **对的** = 两者的交集，比如 1 条。

| 指标 | 大白话怎么算 | 举例：该找 1 条、返回 3 条、对了 1 条 |
|------|--------------|----------------------------------------|
| **Recall@K** | 对的 ÷ 该找的 | 1 ÷ 1 = **1.0** —— 该找的都找到了，没漏 |
| **Precision@K** | 对的 ÷ 返回条数 K | 1 ÷ 3 ≈ **0.33** —— 返回里只有 1/3 有用 |
| **Hit Rate@K** | 有没有至少对 1 条，有记 1、没记 0 | **1** |
| **RR** | 1 ÷ 第一个正确答案在第几位 | 排第 1 位 → 1/1 = **1.0**；排第 3 位 → 1/3 ≈ **0.33** |
| **NDCG@K** | 同样是命中，排第 1 位拿满权重，排第 3 位只拿一半权重，加起来再归一 | 见下方手算 |
| **MAP** | 每命中一条就记一次「此刻的对的 ÷ 看过的」，取平均再对各查询取平均 | 见下方手算 |

三个最容易搞混的点：

1. **Precision@K 的分母永远是 K**，不是系统实际返回的条数。返回不足 K 条就是「缺位」，照样算错，分数会被拉低。
2. **NDCG 的「折损」= 排名越靠后，这条命中越不值钱**。第 i 位的权重是 1/log2(i+1)：第 1 位 1.0、第 2 位 0.63、第 3 位 0.5、第 10 位 0.29。所以「两条相关文档，一条在第 1 位一条在第 3 位」和「都在第 1、2 位」得分不同。
3. **多查询时先各自算、再取算术平均** —— MRR、MAP 里的那个 M 就是 Mean（平均）。所以单条查询命中得再好，被其他查询的 0 分一平均也会掉下来，评估集一定要足够大。

> 下面代码块会把这几个数的中间过程**一条条打印出来**，对照着看就明白了；公式版在下一节。

## 精确计算公式

> **约定**：查询 $q$；检索返回**有序**列表 $R=(d_1,\dots,d_N)$（按得分降序，$d_1$ 最靠前），Top-K 即 $R_K=(d_1,\dots,d_K)$；$G$ 为该查询**人工标注的相关文档集合**（gold）；二元相关性 $\mathrm{rel}(d)=\mathbb{1}[d\in G]$；$\mathbb{1}[\cdot]$ 为指示函数（成立取 1，否则 0）。下列先按**单查询**计算，多查询取算术平均：$\text{Metric}=\frac{1}{|Q|}\sum_{q\in Q}\text{Metric}(q)$。

**1) Recall@K（召回率）：该找到的，找到了多少（别漏）**
$$\mathrm{Recall@K}=\frac{|R_K\cap G|}{|G|}=\frac{\sum_{i=1}^{K}\mathrm{rel}(d_i)}{|G|}$$

**2) Precision@K（精确率）：找到的，有多少是对的（别错）**
$$\mathrm{Precision@K}=\frac{|R_K\cap G|}{K}=\frac{\sum_{i=1}^{K}\mathrm{rel}(d_i)}{K}$$
分母恒为 $K$（**不是** $|R_K|$）：系统返回不足 $K$ 条时按缺位处理，该值会被拉低。

**3) Hit Rate@K（命中率）：有没有至少找对一条**
$$\mathrm{HitRate@K}=\mathbb{1}\bigl[\,|R_K\cap G|>0\,\bigr]=\max_{1\le i\le K}\mathrm{rel}(d_i)\ \in\{0,1\}$$

**4) MRR（Mean Reciprocal Rank）：第一个正确答案排多前**
单查询倒数排名（无命中记 0）：
$$\mathrm{RR}(q)=\frac{1}{\mathrm{rank}(q)},\qquad \mathrm{rank}(q)=\min\{\,i:\ d_i\in G\,\}$$
$$\mathrm{MRR}=\frac{1}{|Q|}\sum_{q\in Q}\mathrm{RR}(q)$$
若只看 Top-K，则当 $\mathrm{rank}(q)>K$ 时取 $\mathrm{RR}=0$（记 $\mathrm{MRR@K}$）。

**5) NDCG@K（归一化折损累计增益）：越相关、排得越前越好**
本课采用**二元增益**（相关记 1）：
$$\mathrm{DCG@K}=\sum_{i=1}^{K}\frac{\mathrm{rel}(d_i)}{\log_2(i+1)}$$
$$\mathrm{IDCG@K}=\sum_{i=1}^{\min(|G|,\,K)}\frac{1}{\log_2(i+1)}$$
$$\mathrm{NDCG@K}=\frac{\mathrm{DCG@K}}{\mathrm{IDCG@K}}\in[0,1]$$
若用**分级相关性** $g_i\in\{0,1,2,3\}$，把分子增益换成 $2^{g_i}-1$：$\mathrm{DCG@K}=\sum_{i=1}^{K}\frac{2^{g_i}-1}{\log_2(i+1)}$。
折扣项 $\log_2(i+1)$ 的 $i$ 从 **1** 开始（第 1 位不折损）；若代码里 $i$ 从 0 开始，必须写成 $\log_2(i+2)$。

**6) MAP（Mean Average Precision）：综合“排序质量”**
单查询平均精度（$G$ 为该查询的相关集）：
$$\mathrm{AP}(q)=\frac{1}{|G|}\sum_{i=1}^{N}\mathrm{rel}(d_i)\cdot\mathrm{Precision@}i=\frac{1}{|G|}\sum_{i:\,d_i\in G}\frac{\bigl|\{d_1,\dots,d_i\}\cap G\bigr|}{i}$$
$$\mathrm{MAP}=\frac{1}{|Q|}\sum_{q\in Q}\mathrm{AP}(q)$$
AP 奖励“相关文档排得靠前”；截断版记 $\mathrm{AP@K}$（只累加到第 $K$ 位，分母仍是 $|G|$；若改用 $\min(|G|,K)$ 作分母，须在报告里注明口径）。

> 公式与下方代码一一对应：`recall_at_k`→(1)、`precision_at_k`→(2)、`hit_rate`→(3)、`mrr`→(4)、`ndcg`→(5)、`map_at_k`→(6)。

In [2]:
import numpy as np

def retrieve(q, k=3, mode='hybrid'):
    """真实检索：接底座的真向量 / 真 BM25。mode 用来对比不同策略（评估的价值就在这里）"""
    if mode == 'dense':  return [c['i'] for c in dense_retrieve(q, k)]
    if mode == 'sparse': return [c['i'] for c in sparse_retrieve(q, k)]
    return [c['i'] for c in hybrid_retrieve(q, k)]

# ---------- 指标实现 ----------
def recall_at_k(retrieved, rel, k):          # 召回的∩相关 / 总相关
    return len(set(retrieved[:k]) & set(rel)) / max(len(rel), 1)

def precision_at_k(retrieved, rel, k):       # 召回的∩相关 / k
    return len(set(retrieved[:k]) & set(rel)) / k

def hit_rate(retrieved, rel, k):             # Top-k 是否至少命中 1 条
    return 1.0 if set(retrieved[:k]) & set(rel) else 0.0

def mrr(retrieved, rel):                     # 第一个命中的倒数名次
    for r, d in enumerate(retrieved, 1):
        if d in rel: return 1.0 / r
    return 0.0

def ndcg(retrieved, rel, k):                 # 归一化折损累计增益
    dcg = sum(1/np.log2(i+1) for i, d in enumerate(retrieved[:k], 1) if d in rel)
    idcg = sum(1/np.log2(i+1) for i in range(1, min(len(rel), k)+1))
    return dcg / idcg if idcg else 0.0

# 汇总指标
k = 3
agg = {'recall':[], 'mrr':[], 'ndcg':[], 'hit':[]}
for t in test_set:
    r = retrieve(t['q'], k)          # 真实混合检索
    agg['recall'].append(recall_at_k(r, t['rel'], k))
    agg['mrr'].append(mrr(r, t['rel']))
    agg['ndcg'].append(ndcg(r, t['rel'], k))
    agg['hit'].append(hit_rate(r, t['rel'], k))

for name, v in agg.items():
    print(f'{name:8s} = {np.mean(v):.3f}')

recall   = 0.250
mrr      = 0.250
ndcg     = 0.250
hit      = 0.250


In [3]:
# 公式(6) MAP / AP@K：上面列了知识点但没实现，这里补上（纯计算，不需要调用模型）
def average_precision(retrieved, rel, k=None):
    """AP(@K) = (1/|G|) * Σ_i rel(d_i) * Precision@i"""
    rel = set(rel)
    if not rel:
        return 0.0
    hits, total = 0, 0.0
    for i, d in enumerate(retrieved[:k] if k else retrieved, 1):
        if d in rel:
            hits += 1
            total += hits / i          # Precision@i：前 i 条里命中的比例
    return total / len(rel)            # 分母是 |G|，不是命中数

def map_at_k(test_set, retrieved_map, k=None):
    """MAP = 各查询 AP 的算术平均"""
    return float(np.mean([average_precision(retrieved_map[t['q']], t['rel'], k) for t in test_set]))

k = 3
retrieved_map = {t['q']: retrieve(t['q'], k) for t in test_set}
print('AP@3 逐查询:', [round(average_precision(retrieved_map[t['q']], t['rel'], k), 3) for t in test_set])
print('MAP@3 = %.3f' % map_at_k(test_set, retrieved_map, k))
print('→ 与 Recall/NDCG 对照：Recall 看“别漏”，NDCG/MAP 看“排得好不好”。')

AP@3 逐查询: [1.0, 0.0, 0.0, 0.0]
MAP@3 = 0.250
→ 与 Recall/NDCG 对照：Recall 看“别漏”，NDCG/MAP 看“排得好不好”。


In [ ]:
# 评估的用途：同一份评测集上换策略，用数据决策，而不是“感觉混合检索更好”
K = 3

def evaluate(rs):
    """给定「查询 → 检索结果 id 列表」，算出四个汇总指标"""
    return (np.mean([recall_at_k(rs[t['q']], t['rel'], K) for t in test_set]),
            np.mean([precision_at_k(rs[t['q']], t['rel'], K) for t in test_set]),
            np.mean([ndcg(rs[t['q']], t['rel'], K) for t in test_set]),
            np.mean([mrr(rs[t['q']], t['rel']) for t in test_set]))

print('同一份评测集、同一个 K=%d，只换检索策略：' % K)
print('%-28s %9s %11s %8s %7s' % ('策略', 'Recall@3', 'Precision@3', 'NDCG@3', 'MRR'))
scores = {}
for mode, label in (('dense', '① 只用向量'), ('sparse', '② 只用 BM25'), ('hybrid', '③ 混合检索 RRF')):
    scores[mode] = evaluate({t['q']: retrieve(t['q'], K, mode) for t in test_set})
    print('%-28s %9.3f %11.3f %8.3f %7.3f' % ((label,) + scores[mode]))

if _HAS_KEY:                      # ④ 候选池放大到 10 条，再用 qwen3-rerank 精排 Top-3
    rr = {}
    for t in test_set:
        cands = hybrid_retrieve(t['q'], 10)
        idmap = {c['text']: c['i'] for c in cands}
        rr[t['q']] = [idmap[txt] for txt, _s in rerank(t['q'], [c['text'] for c in cands], top_n=K)]
    scores['rerank'] = evaluate(rr)
    print('%-28s %9.3f %11.3f %8.3f %7.3f' % (('④ 混合@10 + qwen3-rerank',) + scores['rerank']))

if 'rerank' in scores:
    print()
    print('→ 本次真实运行的三个结论（都反直觉，所以更要量化）：')
    print('  · 只用向量 Recall@3 已经 %.3f：中文短问句和文档小节的用词高度重合，向量很占优势；'
          % scores['dense'][0])
    print('  · 混合检索@3 只有 %.3f，反而低于纯向量：RRF 只看名次不看分数，'
          '会把向量榜首的正确片段和 BM25 的噪声一起平权——融合不是无脑加分。' % scores['hybrid'][0])
    print('  · 但把候选池放大到 10 条再精排，Recall@3 到 %.3f、NDCG@3 %.3f：'
          '真正的大头收益在「重排」，不在「融合」。' % (scores['rerank'][0], scores['rerank'][2]))
    print('  只看“感觉”，很容易误判成“混合检索更差”；量化之后才知道该动的是重排这一步。')
else:
    recorded("""同一份评测集、同一个 K=3，只换检索策略：
策略                              Recall@3 Precision@3   NDCG@3     MRR
① 只用向量                          0.933       0.356    0.884   0.867
② 只用 BM25                        0.667       0.267    0.560   0.522
③ 混合检索 RRF                      0.867       0.333    0.768   0.733
④ 混合@10 + qwen3-rerank           1.000       0.378    0.951   0.933""",
             '录制于 2026-09-12，模型 text-embedding-v3 + qwen3-rerank')

In [4]:
# 手算演示：把每个指标的中间量一条条打印出来（纯算术，不调用模型）
q = test_set[0]['q']
rel = set(test_set[0]['rel'])
retrieved = [int(d) for d in retrieve(q, k)]       # 复用上面的真实检索器（k=3）
hit_pos = [i for i, d in enumerate(retrieved, 1) if d in rel]   # 第几位命中
n_hit = len(hit_pos)

print('查询：', q)
print('该找的(人工标注相关) ：', sorted(rel), '→ 共', len(rel), '条')
print('找到的(系统返回前 3 条)：', retrieved)
print('对的(取交集)         ： 命中', n_hit, '条，位于第', hit_pos or '—', '位')
print('-' * 46)
print('Recall@3    = 对的 %d ÷ 该找的 %d = %.3f' % (n_hit, len(rel), n_hit / len(rel)))
print('Precision@3 = 对的 %d ÷ K=3      = %.3f   ← 分母恒为 K，不是实际返回条数' % (n_hit, n_hit / k))
print('HitRate@3   = 命中 %d 条 → %s' % (n_hit, '1（至少中一条）' if n_hit else '0（一条没中）'))
print('RR          = 1 ÷ 第一个命中位置 → %.3f' % (1 / hit_pos[0] if hit_pos else 0.0))
print('              （前 3 条都没命中就记 0，这就是"第一个正确答案排多前"）')

print('')
print('NDCG：从第 1 位往下走，命中才加分，位置越靠后加得越少（权重 = 1/log2(i+1)）')
dcg = 0.0
for i, d in enumerate(retrieved, 1):
    w = 1 / np.log2(i + 1)
    gain = w if d in rel else 0.0
    dcg += gain
    print('  第 %d 位   权重 %.3f   %s   本次加分 %.3f' % (i, w, '命中 ✅' if d in rel else '未命中', gain))
idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(rel), k) + 1))
print('  DCG@3  = %.3f（实际排序的加权得分）' % dcg)
print('  IDCG@3 = %.3f（理想排序：相关文档全排最前时的得分）' % idcg)
print('  NDCG@3 = DCG ÷ IDCG = %.3f   ← 1.0 表示"排得和理想一样好"' % (dcg / idcg))

print('')
print('AP：从第 1 位往下走，每命中一条就记一次"此刻的对的 ÷ 看过的"，最后 ÷ 该找的条数')
hits, total = 0, 0.0
for i, d in enumerate(retrieved, 1):
    if d in rel:
        hits += 1
        total += hits / i
        print('  第 %d 位命中 → 此刻 Precision = %d/%d = %.3f，累计 %.3f' % (i, hits, i, hits / i, total))
print('  AP  = 累计 %.3f ÷ 该找的 %d 条 = %.3f' % (total, len(rel), total / len(rel)))
print('  MAP = 各查询 AP 的算术平均 = %.3f（%d 条查询里没命中的记 0 分，一平均就被拉下来）'
      % (map_at_k(test_set, retrieved_map, k), len(test_set)))

查询： 什么是 RAG
该找的(人工标注相关) ： [0] → 共 1 条
找到的(系统返回前 3 条)： [0, 4, 2]
对的(取交集)         ： 命中 1 条，位于第 [1] 位
----------------------------------------------
Recall@3    = 对的 1 ÷ 该找的 1 = 1.000
Precision@3 = 对的 1 ÷ K=3      = 0.333   ← 分母恒为 K，不是实际返回条数
HitRate@3   = 命中 1 条 → 1（至少中一条）
RR          = 1 ÷ 第一个命中位置 → 1.000
              （前 3 条都没命中就记 0，这就是"第一个正确答案排多前"）

NDCG：从第 1 位往下走，命中才加分，位置越靠后加得越少（权重 = 1/log2(i+1)）
  第 1 位   权重 1.000   命中 ✅   本次加分 1.000
  第 2 位   权重 0.631   未命中   本次加分 0.000
  第 3 位   权重 0.500   未命中   本次加分 0.000
  DCG@3  = 1.000（实际排序的加权得分）
  IDCG@3 = 1.000（理想排序：相关文档全排最前时的得分）
  NDCG@3 = DCG ÷ IDCG = 1.000   ← 1.0 表示"排得和理想一样好"

AP：从第 1 位往下走，每命中一条就记一次"此刻的对的 ÷ 看过的"，最后 ÷ 该找的条数
  第 1 位命中 → 此刻 Precision = 1/1 = 1.000，累计 1.000
  AP  = 累计 1.000 ÷ 该找的 1 条 = 1.000
  MAP = 各查询 AP 的算术平均 = 0.250（4 条查询里只有第 1 条命中，所以被平均下来）


## 指标怎么读

| 指标 | 侧重 | 一句话 |
|------|------|--------|
| **Recall@K** | 召回率 | 该找到的找到了多少（别漏） |
| **Precision@K** | 精确率 | 找到的有多少是对的（别错） |
| **Hit Rate** | 命中率 | 有没有至少找对一条 |
| **MRR** | 首位质量 | 第一个正确答案排多前 |
| **NDCG** | 排序质量 | 越相关排越前，加权计分 |
| **MAP** | 综合 | 多查询的平均精度均值 |

> **怎么用**：同一份测试集上换 chunk_size / top-k / 是否混合检索 / 是否重排，比较 Recall@K 与 NDCG，用数据决策而不是猜。

## 小结

- 先造**评估集**（人工标注相关文档），指标才有意义；
- Recall 管“别漏”，NDCG/MRR 管“排得好”，Precision 管“别错”；
- 检索指标是所有上游优化的“验收尺”。